# 01 — RFP automation system architecture

> **Applied Labs** · **Domain**: PE · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/11_proposal_rfp_automation/01_architecture.ipynb)

**Prerequisites**:
- `00_first_principles.ipynb` — understanding of RFP response challenges
- Familiarity with RAG patterns and embedding-based retrieval

**What you will learn**:
- How to architect the full RFP automation pipeline
- How to extract structured questions from varied RFP formats
- How to build a RAG-based answer knowledge base with metadata filtering
- How to design context-aware answer adaptation prompts
- How to check cross-section consistency programmatically
- How to assemble a complete proposal from adapted answers

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "chromadb>=0.4.0"

import os, re, json, textwrap
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — System architecture

The RFP automation system is a **multi-stage pipeline** that transforms a raw
RFP document into a complete, consistent proposal:

```
┌──────────────┐    ┌───────────────────┐    ┌──────────────────┐
│ RFP Document │───▶│ Question Extractor │───▶│ Answer Retriever │
│  (PDF/DOCX)  │    │  (parse & split)   │    │   (RAG + KB)     │
└──────────────┘    └───────────────────┘    └────────┬─────────┘
                                                       │
                                                       ▼
┌──────────────┐    ┌───────────────────┐    ┌──────────────────┐
│   Proposal   │◀───│  Consistency      │◀───│ Context Adapter  │
│  Assembler   │    │   Checker         │    │  (LLM + rules)   │
└──────┬───────┘    └───────────────────┘    └──────────────────┘
       │
       ▼
┌──────────────────────────────────────────────────────────────┐
│  Final Proposal: Exec Summary + Answers + Compliance Matrix  │
└──────────────────────────────────────────────────────────────┘
```

Each stage is independently testable and composable. Let's define the
interfaces between stages.

In [ ]:
@dataclass
class ExtractedQuestion:
    """Output of the Question Extractor stage."""
    id: str
    text: str
    category: str
    is_multipart: bool = False
    sub_questions: List[str] = field(default_factory=list)

@dataclass
class RetrievedAnswer:
    """Output of the Answer Retriever stage."""
    question_id: str
    past_answer: str
    relevance_score: float
    source_industry: str
    source_date: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class AdaptedAnswer:
    """Output of the Context Adapter stage."""
    question_id: str
    original_answer: str
    adapted_answer: str
    adaptations_made: List[str] = field(default_factory=list)

@dataclass
class ConsistencyResult:
    """Output of the Consistency Checker stage."""
    is_consistent: bool
    contradictions: List[Dict[str, str]] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)

@dataclass
class Proposal:
    """Final assembled proposal."""
    title: str
    executive_summary: str
    answers: List[AdaptedAnswer]
    compliance_matrix: List[Dict[str, str]]
    consistency: ConsistencyResult

print("Pipeline data contracts defined ✓")
print(f"  Stages: Extraction → Retrieval → Adaptation → Consistency → Assembly")

## Section 2 — RFP question extraction

RFPs arrive in various formats: numbered lists, tables, multi-part questions,
and free-form paragraphs with embedded requirements. The extractor must
handle all of these and produce a clean list of `ExtractedQuestion` objects.

Key challenges:
- Numbered lists (1. / 1.1 / a.) with varying nesting
- Table-formatted questions (row = question)
- Multi-part questions ("Describe X. Also provide Y.")
- Implicit requirements buried in context paragraphs

In [ ]:
class RFPQuestionExtractor:
    """Extract structured questions from raw RFP text.

    Handles numbered lists, multi-part questions, and table-style formats.
    """

    # Patterns for numbered items
    NUMBERED_PATTERN = re.compile(
        r"^\s*(?:(?P<num>\d+(?:\.\d+)*)[\.\)\s]|(?P<letter>[a-z])[\.\)\s])"
        r"\s*(?P<text>.+)",
        re.MULTILINE
    )
    # Pattern for multi-part detection
    MULTIPART_SPLITTERS = re.compile(
        r"(?:Additionally|Also|Furthermore|In addition|Please also)[,:]?\s",
        re.IGNORECASE
    )

    CATEGORY_KEYWORDS: Dict[str, List[str]] = {
        "security": ["encrypt", "security", "access control", "authentication", "vulnerability"],
        "compliance": ["compliance", "certif", "hipaa", "sox", "pci", "gdpr", "audit"],
        "technical": ["architecture", "api", "integration", "uptime", "disaster", "backup"],
        "pricing": ["price", "pricing", "cost", "fee", "discount", "subscription"],
        "support": ["support", "response time", "sla", "help desk", "customer success"],
    }

    def classify_category(self, text: str) -> str:
        """Classify question into a category based on keyword matching."""
        text_lower = text.lower()
        scores: Dict[str, int] = {}
        for cat, keywords in self.CATEGORY_KEYWORDS.items():
            scores[cat] = sum(1 for kw in keywords if kw in text_lower)
        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else "general"

    def detect_multipart(self, text: str) -> Tuple[bool, List[str]]:
        """Detect and split multi-part questions."""
        parts = self.MULTIPART_SPLITTERS.split(text)
        parts = [p.strip() for p in parts if len(p.strip()) > 10]
        return len(parts) > 1, parts

    def extract(self, raw_text: str) -> List[ExtractedQuestion]:
        """Extract questions from raw RFP text.

        Returns list of ExtractedQuestion with categories and multi-part handling.
        """
        questions: List[ExtractedQuestion] = []
        for match in self.NUMBERED_PATTERN.finditer(raw_text):
            num = match.group("num") or match.group("letter")
            text = match.group("text").strip()
            is_multi, sub_parts = self.detect_multipart(text)
            category = self.classify_category(text)
            questions.append(ExtractedQuestion(
                id=f"Q{num}",
                text=text,
                category=category,
                is_multipart=is_multi,
                sub_questions=sub_parts if is_multi else [],
            ))
        return questions

# Test with a sample RFP excerpt
sample_rfp_text = """
1. Describe your data encryption approach for data at rest and in transit.
2. What compliance certifications does your platform hold? Additionally, provide your most recent audit dates.
3. Describe your disaster recovery and business continuity plan.
4. What is your guaranteed uptime SLA?
5. How do you handle HIPAA-protected health information? Also describe your BAA process.
6. Describe your pricing model and any volume discounts.
7. What support tiers do you offer and their response times?
8. Describe your API integration capabilities. Furthermore, list supported authentication methods.
"""

extractor = RFPQuestionExtractor()
extracted = extractor.extract(sample_rfp_text)

print(f"Extracted {len(extracted)} questions:\n")
for q in extracted:
    multi_tag = " [MULTI-PART]" if q.is_multipart else ""
    print(f"  {q.id} ({q.category}){multi_tag}: {q.text[:75]}")
    if q.sub_questions:
        for i, sq in enumerate(q.sub_questions):
            print(f"      Part {i+1}: {sq[:70]}")

## Section 3 — Answer knowledge base (RAG)

The answer retriever uses **RAG** over a knowledge base of past proposal Q&A
pairs. Each entry is embedded and stored with rich metadata for filtering.

Design decisions:
- **Chunking**: By Q&A pair (not arbitrary text chunks)
- **Embeddings**: sentence-transformers for semantic similarity
- **Metadata**: industry, date, client size, topic, win/loss status
- **Filtering**: Pre-filter by category, then rank by embedding similarity

In [ ]:
from sentence_transformers import SentenceTransformer

@dataclass
class KBEntry:
    """A single knowledge base entry — one past Q&A pair with metadata."""
    question: str
    answer: str
    industry: str
    client_size: str
    date: str
    topic: str
    won: bool = True

class AnswerKnowledgeBase:
    """Embedding-based answer retriever with metadata filtering.

    Uses sentence-transformers for semantic search over past Q&A pairs.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2") -> None:
        self.model = SentenceTransformer(model_name)
        self.entries: List[KBEntry] = []
        self.embeddings: Optional[np.ndarray] = None

    def add_entries(self, entries: List[KBEntry]) -> None:
        """Add entries and compute embeddings."""
        self.entries.extend(entries)
        texts = [e.question + " " + e.answer for e in self.entries]
        self.embeddings = self.model.encode(texts, normalize_embeddings=True)
        print(f"KB: indexed {len(self.entries)} entries, dim={self.embeddings.shape[1]}")

    def retrieve(self, query: str, top_k: int = 3,
                 category_filter: Optional[str] = None) -> List[Tuple[KBEntry, float]]:
        """Retrieve top-k most relevant past answers.

        Args:
            query: The current RFP question.
            top_k: Number of results to return.
            category_filter: Optional topic filter.

        Returns:
            List of (KBEntry, score) tuples sorted by relevance.
        """
        q_emb = self.model.encode([query], normalize_embeddings=True)
        scores = (self.embeddings @ q_emb.T).flatten()

        indices = list(range(len(self.entries)))
        if category_filter:
            indices = [i for i in indices
                       if self.entries[i].topic.lower() == category_filter.lower()]

        ranked = sorted(indices, key=lambda i: scores[i], reverse=True)
        return [(self.entries[i], float(scores[i])) for i in ranked[:top_k]]

# Build a small knowledge base
kb_entries: List[KBEntry] = [
    KBEntry("How do you encrypt data?",
            "AES-256 at rest, TLS 1.3 in transit. Key management via AWS KMS.",
            "healthcare", "enterprise", "2024-01", "security"),
    KBEntry("What security certifications do you hold?",
            "SOC 2 Type II, ISO 27001, HITRUST CSF, FedRAMP Moderate.",
            "government", "enterprise", "2024-02", "compliance"),
    KBEntry("Describe your backup strategy.",
            "Daily backups, 15-min RPO, 1-hr RTO. Geo-redundant across 3 regions.",
            "fintech", "mid-market", "2023-11", "technical"),
    KBEntry("What is your uptime guarantee?",
            "99.9% SLA with tiered service credits: 10% at 99.5%, 25% below 99%.",
            "retail", "enterprise", "2024-01", "technical"),
    KBEntry("Are you HIPAA compliant?",
            "Yes. We execute BAAs, encrypt all PHI, maintain audit logs, annual pen tests.",
            "healthcare", "enterprise", "2024-03", "compliance"),
    KBEntry("Describe your API capabilities.",
            "RESTful API with OpenAPI 3.0 spec. SDKs for Python, JS, Java. Rate limit: 1000 req/min.",
            "technology", "startup", "2023-09", "technical"),
]

kb = AnswerKnowledgeBase()
kb.add_entries(kb_entries)

# Test retrieval
test_query = "How do you protect sensitive patient data during transmission?"
results = kb.retrieve(test_query, top_k=3)
print(f"\nQuery: {test_query}\n")
for entry, score in results:
    print(f"  [{score:.3f}] ({entry.industry}, {entry.topic}) {entry.question}")
    print(f"           → {entry.answer[:80]}...\n")

## Section 4 — Context-aware answer adaptation

The adapter transforms a retrieved past answer into a context-appropriate
response. The **adaptation prompt** includes:

1. The **past answer** as a starting point
2. The **current RFP context** (industry, company size, tone)
3. The **specific requirements** from this RFP question
4. **Adaptation rules** (what to emphasize, add, or remove)

Let's design the prompt template and build the adapter.

In [ ]:
ADAPTATION_PROMPT_TEMPLATE: str = """You are an expert proposal writer.

## Task
Adapt the PAST ANSWER to fit the CURRENT RFP CONTEXT. Maintain factual accuracy
while tailoring language, emphasis, and specific details to the target audience.

## Past Answer
{past_answer}

## Current RFP Context
- Industry: {industry}
- Company size: {company_size}
- Specific requirements: {requirements}
- Tone: {tone}

## Adaptation Rules
1. Add industry-specific terminology and compliance references
2. Adjust detail level for company size (enterprise = more governance detail)
3. Address every specific requirement mentioned
4. Maintain all factual claims from the past answer
5. Do NOT invent capabilities not present in the past answer

## Adapted Answer
"""

@dataclass
class AdaptationConfig:
    """Configuration for answer adaptation."""
    industry: str
    company_size: str
    requirements: List[str]
    tone: str = "professional"

class AnswerAdapter:
    """Adapts past answers to current RFP context.

    In production, calls an LLM. Here we demonstrate the prompt construction
    and rule-based adaptation logic.
    """

    INDUSTRY_TERMS: Dict[str, List[str]] = {
        "healthcare": ["HIPAA", "PHI", "BAA", "ePHI", "covered entity"],
        "fintech": ["PCI-DSS", "SOX", "GLBA", "transaction security"],
        "government": ["FedRAMP", "FISMA", "NIST 800-53", "ATO"],
        "education": ["FERPA", "student data", "COPPA"],
    }

    def build_prompt(self, past_answer: str, config: AdaptationConfig) -> str:
        """Build the adaptation prompt for the LLM."""
        return ADAPTATION_PROMPT_TEMPLATE.format(
            past_answer=past_answer,
            industry=config.industry,
            company_size=config.company_size,
            requirements=", ".join(config.requirements),
            tone=config.tone,
        )

    def analyze_adaptation_needs(self, past_answer: str,
                                  config: AdaptationConfig) -> List[str]:
        """Determine what adaptations are needed (rule-based analysis)."""
        needs: List[str] = []
        # Check for missing industry terms
        terms = self.INDUSTRY_TERMS.get(config.industry, [])
        missing = [t for t in terms if t.lower() not in past_answer.lower()]
        if missing:
            needs.append(f"Add {config.industry} terms: {missing[:3]}")
        # Check for unaddressed requirements
        for req in config.requirements:
            if req.lower() not in past_answer.lower():
                needs.append(f"Address requirement: {req}")
        # Size-based adjustments
        if config.company_size == "enterprise":
            needs.append("Add governance and audit trail details")
        elif config.company_size == "startup":
            needs.append("Simplify; emphasize quick deployment")
        return needs

# Demo
adapter = AnswerAdapter()
past = "AES-256 at rest, TLS 1.3 in transit. Key management via AWS KMS."

configs = [
    AdaptationConfig("healthcare", "enterprise", ["HIPAA", "BAA", "PHI encryption"]),
    AdaptationConfig("fintech", "mid-market", ["PCI-DSS Level 1", "transaction logs"]),
    AdaptationConfig("government", "enterprise", ["FedRAMP", "NIST 800-53"]),
]

for cfg in configs:
    needs = adapter.analyze_adaptation_needs(past, cfg)
    print(f"Context: {cfg.industry} / {cfg.company_size}")
    print(f"  Adaptations needed ({len(needs)}):")
    for n in needs:
        print(f"    → {n}")
    print()

# Show prompt for first config
prompt = adapter.build_prompt(past, configs[0])
print("Sample adaptation prompt (truncated):")
print(prompt[:500])

## Section 5 — Consistency checking

After adapting all answers, we must verify that **factual claims are
consistent** across sections. The checker:

1. **Extracts claims** — uptime percentages, response times, certifications,
   pricing figures
2. **Groups by type** — all uptime claims together, all pricing together
3. **Detects conflicts** — different values for the same claim type
4. **Suggests resolution** — which value to standardize on

In [ ]:
class ConsistencyChecker:
    """Check cross-section consistency of proposal answers.

    Extracts factual claims and detects contradictions.
    """

    CLAIM_PATTERNS: Dict[str, re.Pattern] = {
        "uptime": re.compile(r"(\d{2,3}\.\d+)\s*%\s*(?:uptime|availability)", re.I),
        "rto": re.compile(r"(\d+)\s*-?\s*(?:hour|hr|minute|min)\s*RTO", re.I),
        "rpo": re.compile(r"(\d+)\s*-?\s*(?:hour|hr|minute|min)\s*RPO", re.I),
        "pricing": re.compile(r"\$([\d,]+(?:\.\d{2})?)\s*(?:per|/)", re.I),
    }
    CERT_PATTERN = re.compile(r"(SOC\s*2|ISO\s*27001|HITRUST|FedRAMP|HIPAA|PCI-DSS)", re.I)

    def extract_claims(self, section_id: str,
                       text: str) -> List[Dict[str, str]]:
        """Extract factual claims from a section of text."""
        claims: List[Dict[str, str]] = []
        for claim_type, pattern in self.CLAIM_PATTERNS.items():
            for m in pattern.finditer(text):
                claims.append({
                    "section": section_id,
                    "type": claim_type,
                    "value": m.group(1) if m.lastindex else m.group(0),
                    "raw": m.group(0),
                })
        for m in self.CERT_PATTERN.finditer(text):
            claims.append({"section": section_id, "type": "certification",
                          "value": m.group(1).upper(), "raw": m.group(0)})
        return claims

    def check(self, sections: Dict[str, str]) -> ConsistencyResult:
        """Check all sections for consistency.

        Args:
            sections: Dict mapping section_id to answer text.

        Returns:
            ConsistencyResult with contradictions and warnings.
        """
        all_claims: List[Dict[str, str]] = []
        for sid, text in sections.items():
            all_claims.extend(self.extract_claims(sid, text))

        grouped: Dict[str, List[Dict[str, str]]] = defaultdict(list)
        for c in all_claims:
            grouped[c["type"]].append(c)

        contradictions: List[Dict[str, str]] = []
        for ctype, claims in grouped.items():
            if ctype == "certification":
                continue  # certs can differ by section context
            values = set(c["value"] for c in claims)
            if len(values) > 1:
                contradictions.append({
                    "type": ctype,
                    "values": str(values),
                    "sections": str([c["section"] for c in claims]),
                    "suggestion": f"Standardize {ctype} to one value",
                })

        return ConsistencyResult(
            is_consistent=len(contradictions) == 0,
            contradictions=contradictions,
            warnings=[f"Found {len(all_claims)} claims across {len(sections)} sections"],
        )

# Test with intentionally inconsistent sections
checker = ConsistencyChecker()
test_sections: Dict[str, str] = {
    "Q3_DR": "We guarantee 99.9% uptime. RTO is 1 hour, RPO is 15 minute.",
    "Q4_SLA": "Our 99.99% availability SLA is backed by service credits.",
    "Q7_Support": "Enterprise support includes 99.9% uptime guarantee and 2 hour RTO.",
}

result = checker.check(test_sections)
print(f"Consistent: {result.is_consistent}")
print(f"Warnings: {result.warnings}\n")
for con in result.contradictions:
    print(f"  ⚠ {con['type']}: values={con['values']}")
    print(f"    Sections: {con['sections']}")
    print(f"    Fix: {con['suggestion']}\n")

## Section 6 — Proposal assembly

The final stage combines adapted answers into a structured proposal document.
The assembler generates:

1. **Executive summary** — tailored to the prospect
2. **Structured answers** — formatted by category
3. **Compliance matrix** — requirement → capability → evidence → status
4. **Cover letter** — personalized with prospect details

Let's design the output format and build the assembler.

In [ ]:
class ProposalAssembler:
    """Assemble adapted answers into a complete proposal document."""

    def _group_by_category(self,
                           answers: List[Dict[str, str]]) -> Dict[str, List[Dict[str, str]]]:
        """Group answers by their category."""
        grouped: Dict[str, List[Dict[str, str]]] = defaultdict(list)
        for a in answers:
            grouped[a.get("category", "general")].append(a)
        return dict(grouped)

    def build_compliance_matrix(
        self,
        requirements: List[str],
        answers: List[Dict[str, str]]
    ) -> List[Dict[str, str]]:
        """Build compliance matrix from requirements and answers.

        Maps each requirement to the most relevant answer and status.
        """
        matrix: List[Dict[str, str]] = []
        for req in requirements:
            # Simple keyword matching for demo
            best_answer = "See response document"
            status = "addressed"
            for a in answers:
                if any(word in a.get("text", "").lower()
                       for word in req.lower().split()[:3]):
                    best_answer = a.get("answer", "")[:100]
                    status = "fully_addressed"
                    break
            matrix.append({
                "requirement": req,
                "response_summary": best_answer,
                "status": status,
            })
        return matrix

    def assemble(self, rfp_title: str, prospect: str,
                 answers: List[Dict[str, str]],
                 requirements: List[str],
                 consistency: ConsistencyResult) -> Dict[str, Any]:
        """Assemble the final proposal structure.

        Returns a dict representing the full proposal document.
        """
        grouped = self._group_by_category(answers)
        matrix = self.build_compliance_matrix(requirements, answers)

        proposal: Dict[str, Any] = {
            "title": f"Proposal: {rfp_title}",
            "prospect": prospect,
            "executive_summary": (
                f"We are pleased to submit our response to {prospect}'s "
                f"{rfp_title}. Our solution addresses all {len(answers)} "
                f"requirements with {sum(1 for m in matrix if m['status'] == 'fully_addressed')} "
                f"fully addressed items."
            ),
            "sections": grouped,
            "compliance_matrix": matrix,
            "consistency_check": {
                "passed": consistency.is_consistent,
                "issues": len(consistency.contradictions),
            },
            "total_questions": len(answers),
        }
        return proposal

# Demo assembly
assembler = ProposalAssembler()
demo_answers: List[Dict[str, str]] = [
    {"id": "Q1", "category": "security", "text": "encryption approach",
     "answer": "AES-256 at rest, TLS 1.3 in transit with HIPAA safeguards."},
    {"id": "Q2", "category": "compliance", "text": "compliance certifications",
     "answer": "SOC 2 Type II, ISO 27001, HITRUST CSF."},
    {"id": "Q3", "category": "technical", "text": "disaster recovery",
     "answer": "Geo-redundant DR with 1-hr RTO and 15-min RPO."},
]
demo_reqs = ["Data encryption required", "Must hold SOC 2", "DR plan mandatory"]
demo_consistency = ConsistencyResult(is_consistent=True)

proposal = assembler.assemble(
    "Cloud Platform Vendor Selection",
    "Midwest Healthcare Group",
    demo_answers, demo_reqs, demo_consistency
)

print("Assembled proposal structure:")
print(json.dumps(proposal, indent=2)[:800])
print("\n✓ Proposal assembly complete")

In [ ]:
# End-to-end pipeline simulation — connect all stages
def run_pipeline(rfp_text: str, industry: str,
                 company_size: str) -> Dict[str, Any]:
    """Simulate the full RFP automation pipeline.

    Args:
        rfp_text: Raw RFP text to process.
        industry: Target industry for adaptation.
        company_size: Target company size.

    Returns:
        Dict with pipeline results and timing for each stage.
    """
    import time
    results: Dict[str, Any] = {"stages": {}}

    # Stage 1: Extract
    t0 = time.time()
    questions = extractor.extract(rfp_text)
    results["stages"]["extract"] = {"time_ms": round((time.time() - t0) * 1000, 1),
                                     "questions": len(questions)}

    # Stage 2: Retrieve (simulated)
    t0 = time.time()
    retrievals = {}
    for q in questions:
        hits = kb.retrieve(q.text, top_k=3)
        retrievals[q.id] = hits
    results["stages"]["retrieve"] = {"time_ms": round((time.time() - t0) * 1000, 1),
                                      "avg_score": round(np.mean([h[1] for hits in retrievals.values() for h in hits[:1]]), 3)}

    # Stage 3: Adapt (rule-based for demo)
    t0 = time.time()
    config = AdaptationConfig(industry, company_size, [])
    adaptations = {}
    for q in questions:
        if retrievals.get(q.id):
            past = retrievals[q.id][0][0].answer
            needs = adapter.analyze_adaptation_needs(past, config)
            adaptations[q.id] = {"needs": len(needs), "past_answer": past}
    results["stages"]["adapt"] = {"time_ms": round((time.time() - t0) * 1000, 1),
                                   "total_adaptations": sum(a["needs"] for a in adaptations.values())}

    # Stage 4: Consistency check (simulated)
    results["stages"]["consistency"] = {"time_ms": 5.0, "contradictions": 0}

    # Stage 5: Assembly (simulated)
    results["stages"]["assembly"] = {"time_ms": 2.0, "sections": len(questions)}

    total_ms = sum(s["time_ms"] for s in results["stages"].values())
    results["total_ms"] = round(total_ms, 1)
    return results

pipeline_result = run_pipeline(sample_rfp_text, "healthcare", "enterprise")
print("Pipeline simulation results:\n")
for stage, data in pipeline_result["stages"].items():
    print(f"  {stage:>15}: {data['time_ms']:>6.1f} ms  {data}")
print(f"\n  Total: {pipeline_result['total_ms']} ms")
print("\n✓ Full pipeline runs in under 1 second (excluding LLM calls)")

## Exercises

1. **Add table-format extraction** — Extend `RFPQuestionExtractor` to handle
   pipe-delimited table rows (`| # | Question | Required |`). Test with a
   5-row table RFP format.

2. **Implement metadata-weighted retrieval** — Modify `AnswerKnowledgeBase`
   to boost retrieval scores for entries matching the target industry. Compare
   retrieval quality with and without the boost.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | The pipeline decomposes into 5 stages: extract → retrieve → adapt → check → assemble |
| 2 | Question extraction must handle numbered lists, tables, and multi-part questions |
| 3 | RAG over Q&A pairs with metadata beats generic document retrieval |
| 4 | Adaptation prompts need past answer + context + rules to prevent hallucination |
| 5 | Consistency checking requires structured claim extraction and cross-section comparison |
| 6 | Proposal assembly groups answers by category and generates compliance matrices |

## What's Next

In **02 — Build**, we implement the full pipeline end-to-end: a 30-entry
knowledge base, a 15-question healthcare RFP, retrieval, LLM-powered
adaptation, consistency checking, and final proposal assembly.

---

## References

1. Lewis, P. et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020 — <https://arxiv.org/abs/2005.11401>
2. RFPIO/Responsive — <https://www.responsive.io/>
3. ChromaDB Documentation — <https://docs.trychroma.com/>
4. OpenAI API Documentation — <https://platform.openai.com/docs/>